In [1]:
# Cell 1
%matplotlib inline
import pandas as pd
import geopandas as gpd
from pathlib import Path
from scripts.shared import db_utils
from scripts.shared.db_utils import db_connect

conn = db_connect()
ROOT = Path(db_utils.__file__).parent.parent.parent
print('connected')

connected


In [2]:
# Cell 2 — find the Northern Song name string
# Cliopatria naming can vary; cast a wide net first
sql = """
SELECT DISTINCT name, COUNT(*) AS n_rows
FROM gaz.clio_polities
WHERE name ILIKE '%song%'
GROUP BY name
ORDER BY name
"""
pd.read_sql(sql, conn)

,name,n_rows
0,Liu Song Dynasty,4
1,Northern Song,6
2,Song,8
3,Songhai Empire,8
4,Southern Song,9


In [3]:
# Cell 3 — territorial phases for Northern Song
# Replace the name string below if Cell 2 shows a different exact name
POLITY_NAME = 'Northern Song'   # adjust if needed

sql = f"""
SELECT id, name, fromyear, toyear,
       ROUND(ST_Area(geom::geography) / 1e6) AS area_km2
FROM gaz.clio_polities
WHERE name = '{POLITY_NAME}'
ORDER BY fromyear
"""
phases = pd.read_sql(sql, conn)
print(f'{len(phases)} row(s) found')
phases

,id,name,fromyear,toyear,area_km2
0,4352,Northern Song,961,961,902094.0
1,4367,Northern Song,962,969,902189.0
2,4401,Northern Song,970,979,1579520.0
3,4454,Northern Song,980,989,2765956.0
4,4481,Northern Song,990,1017,2764506.0
5,4683,Northern Song,1018,1027,2764506.0


In [4]:
# Cell 4 — pick year, sanity-check geometry
# 1100 CE preferred (symmetry with Timbuktu Band T fixture); adjust YEAR if not covered
YEAR = 1000

sql = f"""
SELECT id, name, fromyear, toyear,
       ST_IsValid(geom)                          AS is_valid,
       ST_NumGeometries(geom)                    AS n_geoms,
       ST_NPoints(geom)                          AS n_vertices,
       ROUND(ST_Area(geom::geography) / 1e6)     AS area_km2,
       ST_AsText(geom)                           AS geom_wkt
FROM gaz.clio_polities
WHERE name = '{POLITY_NAME}'
  AND fromyear <= {YEAR} AND toyear >= {YEAR}
"""
row = pd.read_sql(sql, conn)
print(f'{len(row)} row(s) match year {YEAR}')
row[['id','name','fromyear','toyear','is_valid','n_geoms','n_vertices','area_km2']]

,id,name,fromyear,toyear,is_valid,n_geoms,n_vertices,area_km2
0,4481,Northern Song,990,1017,True,3,313,2764506.0


In [7]:
# Cell 5 — extract GEOM_WKT; round-trip via DB (parameterised, no string embedding)
assert len(row) == 1, f'Expected 1 match for year {YEAR}, got {len(row)}'
POLITY_ID = int(row.iloc[0]['id'])
FROMYEAR  = int(row.iloc[0]['fromyear'])
TOYEAR    = int(row.iloc[0]['toyear'])
GEOM_WKT  = row.iloc[0]['geom_wkt']

check = conn.execute(
    "SELECT ST_IsValid(ST_GeomFromText(%s, 4326)) AS valid, "
    "length(ST_AsText(ST_GeomFromText(%s, 4326))) AS wkt_len",
    (GEOM_WKT, GEOM_WKT)
).fetchone()

print(f'polity_id={POLITY_ID}  phase={FROMYEAR}–{TOYEAR}  year={YEAR}')
print(f'wkt_len={check[1]:,} chars   round-trip valid={check[0]}')
print(f'Note: Cliopatria Northern Song ends at 1027 CE; using year={YEAR} (phase 990–1017)')

polity_id=4481  phase=990–1017  year=1000
wkt_len=11,696 chars   round-trip valid=True
Note: Cliopatria Northern Song ends at 1027 CE; using year=1000 (phase 990–1017)


In [8]:
# Cell 6 — Part 2: resolve_polygon SQL (exploration before engine.py)
# ST_Intersection computed in geometry space; areas measured geodetically via ::geography
# b.geog is the precomputed geography column on basin06 — consistent with buffer resolver
LEVEL = 6
TABLE = f'basin{LEVEL:02d}'

sql = f"""
WITH polity AS (
    SELECT  ST_GeomFromText(%s, 4326)                         AS geom,
            ST_Area(ST_GeomFromText(%s, 4326)::geography)     AS area_m2
),
intersections AS (
    SELECT  b.hybas_id,
            ST_Area(ST_Intersection(b.geom, p.geom)::geography) AS overlap_m2,
            ST_Area(b.geog)                                       AS basin_area_m2
    FROM    {TABLE} b, polity p
    WHERE   ST_Intersects(b.geom, p.geom)
)
SELECT  i.hybas_id,
        i.overlap_m2 / p.area_m2                AS weight,
        i.overlap_m2 / i.basin_area_m2          AS basin_in_polity_fraction,
        ROUND((i.overlap_m2 / 1e6)::numeric, 2) AS overlap_area_km2
FROM    intersections i, polity p
ORDER BY weight DESC
"""

cur  = conn.execute(sql, (GEOM_WKT, GEOM_WKT))
cols = [d[0] for d in cur.description]
basin_set = pd.DataFrame(cur.fetchall(), columns=cols)
basin_set['hybas_id'] = basin_set['hybas_id'].astype('Int64')

weight_sum = basin_set['weight'].sum()
shortfall  = 1.0 - weight_sum
print(f'n_basins={len(basin_set)}  weight_sum={weight_sum:.6f}  shortfall={shortfall:.6f}')
print()
print(basin_set.head(10).to_string(index=False))

n_basins=376  weight_sum=0.989454  shortfall=0.010546

  hybas_id   weight  basin_in_polity_fraction overlap_area_km2
4060764560 0.014066                  1.000000         38884.88
4060765990 0.013012                  1.000000         35971.67
4060780120 0.012269                  1.000000         33917.96
4060047500 0.012262                  0.984550         33898.28
4060802680 0.010661                  1.000000         29472.77
4060012280 0.010607                  0.999709         29324.34
4060603570 0.010306                  1.000000         28491.09
4060631960 0.010281                  1.000000         28421.14
4060693050 0.010215                  1.000000         28239.36
4060801370 0.010041                  1.000000         27759.51


In [9]:
# Cell 7 — Part 2 acceptance checks
assert len(basin_set) > 0,                         'no basins returned'
assert (basin_set['weight'] > 0).all(),            'zero-weight basins present'
assert (basin_set['weight'] <= 1).all(),           'weight > 1 (impossible)'
assert basin_set['weight'].sum() <= 1.0 + 1e-6,   'weight_sum exceeds 1'
assert (basin_set['basin_in_polity_fraction'] >= 0).all(), 'negative basin_in_polity_fraction'
assert (basin_set['basin_in_polity_fraction'] <= 1 + 1e-6).all(), 'basin_in_polity_fraction > 1'
print('all acceptance checks PASS')
print(f'n_basins={len(basin_set)}  shortfall={shortfall:.4f}')
print(f'basin_in_polity_fraction: min={basin_set["basin_in_polity_fraction"].min():.4f}  '
      f'max={basin_set["basin_in_polity_fraction"].max():.4f}  '
      f'median={basin_set["basin_in_polity_fraction"].median():.4f}')

all acceptance checks PASS
n_basins=376  shortfall=0.0105
basin_in_polity_fraction: min=0.0002  max=1.0000  median=1.0000


In [10]:
# Cell 8 — Part 3: resolve_polity lookup logic (inline before engine.py)
# resolve_polity(name, year, level, conn) → (geom_wkt, basin_set, polity_meta)
# The wrapper adds: single-row assertion, polity_meta dict, call to resolve_polygon

sql = """
SELECT id, name, fromyear, toyear, ST_AsText(geom) AS geom_wkt
FROM gaz.clio_polities
WHERE name = %s AND fromyear <= %s AND toyear >= %s
"""
cur  = conn.execute(sql, (POLITY_NAME, YEAR, YEAR))
rows = cur.fetchall()

assert len(rows) != 0, f'No polity row for name={POLITY_NAME!r} year={YEAR}'
if len(rows) > 1:
    # pick most temporally specific (smallest span) — flag if this fires
    rows = sorted(rows, key=lambda r: r[3] - r[2])
    print(f'WARNING: {len(rows)} rows matched — picked narrowest span ({rows[0][2]}–{rows[0][3]})')

r = rows[0]
polity_meta = {
    'id': int(r[0]), 'name': r[1],
    'fromyear': int(r[2]), 'toyear': int(r[3]), 'year': YEAR
}
wkt = r[4]

assert wkt == GEOM_WKT, 'WKT mismatch vs Cell 5 — lookup inconsistent'
print(f'resolve_polity: name={polity_meta["name"]}  id={polity_meta["id"]}  '
      f'phase={polity_meta["fromyear"]}–{polity_meta["toyear"]}  year={polity_meta["year"]}')
print('Part 3 acceptance: PASS')

resolve_polity: name=Northern Song  id=4481  phase=990–1017  year=1000
Part 3 acceptance: PASS
